# 02 — Price concentration and reaction levels

This notebook returns to **finding candidate levels**, not predicting the next trade.
It uses the **first 200 trades from the same saved AAPL sample** used in notebook 01 (2026-08-20).
The earlier sample contains 2,000 trades; 200 is the explicitly selected prefix here. No new ticker or sample is substituted.

Follow the cells in order:
1. Show the original data and preserve ordering.
2. Calculate our six agreed differences.
3. Aggregate the nine direction combinations at each price.
4. Find activity concentrations along the price axis.
5. Describe their immediate support/resistance evidence.
6. Trace a selected candidate back to its individual trades and plot the result.

No candles, visit IDs, smoothing, price bins, prediction model, or composite strength score are introduced. These are descriptive candidates at the final observed event, not proven trading levels.

In [ ]:
# Basic setup. Generated data and charts must stay outside the repository.
import os
import sys
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path("D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path("D:/TradingML/runtimes")
SOURCE_RUN = RUNTIME_ROOT / "level-book-v8/reaction-demo/20261020T204419563339Z"
EXPECTED_SOURCE_HASH = "b40270ba8fc4055d515fa9061ce5333cce152890de23e249dbbc827e7e0ec70d"
N_EVENTS = 200
# Illustrative, visible thresholds. They are not calibrated or tuned on this sample.
MIN_TRADES = 3           # A candidate must have at least three trades at its exact price.
MIN_DEPARTURES = 3       # Minimum immediate nonzero departures for a directional label.
MIN_REVERSAL_SHARE = 0.60
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required root unavailable"
RUN = RUNTIME_ROOT / "level-book-v8/level-discovery" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
print("Output directory:", RUN)

## 1. Load and verify the original sample
This reads the saved canonical-derived snapshot, not raw SIP flatfiles. We validate its original content hash before taking the first 200 rows. If it is missing or changed, stop rather than silently fetching a different sample.

The original extraction excluded delayed/unknown-clock and condition-ineligible trades. Its audit and provenance are retained. Price is stored in exact units of 1/10,000 dollar. Ordinal is metadata, not a predictor: order remains `(SIP timestamp, ordinal)`, exactly as in notebook 01. Time differences can be zero for equal timestamps.

We slice **before** calculating shifts. Event 201 must not reveal the departure of event 200.

In [ ]:
# Recreate the original loader's hashed records, in their original order.
source = pl.read_parquet(SOURCE_RUN / "source_trades.parquet")
provenance = json.loads((SOURCE_RUN / "source_provenance.json").read_text(encoding="utf-8"))
source_records = source.select("ordinal", "time_us", "price_units", "volume").to_dicts()
# ClickHouse JSON encoded integral sizes as integers; Parquet promoted the
# mixed size column to Float64. Restore only that JSON representation for hashing.
for record in source_records:
    if float(record["volume"]).is_integer():
        record["volume"] = int(record["volume"])
source_hash = hashlib.sha256(json.dumps(source_records, sort_keys=True).encode()).hexdigest()
assert source_hash == provenance["sample_sha256"] == EXPECTED_SOURCE_HASH
assert source.height == 2000 and provenance["ticker"] == "AAPL"
assert provenance["day"] == "2026-08-20"
assert 3 <= N_EVENTS <= source.height
assert source["ordinal"].n_unique() == source.height
assert source.select("time_us", "ordinal").equals(source.sort("time_us", "ordinal").select("time_us", "ordinal"))
raw = source.head(N_EVENTS)
# These are the three original quantities; event and ordinal identify rows.
display(raw.select("event", "ordinal", "time", "price", "volume").head(15))
print("Selected events:", raw.height, "unique prices:", raw["price_units"].n_unique())
print("Time span:", raw["time"][0], "to", raw["time"][-1])
display(pl.DataFrame(provenance["audit"]))

## 2. Our agreed step one: six vectorized differences
For any column x:

- backward = current minus previous;
- forward = next minus current.

Time differences are seconds, price differences dollars, and volume differences changes in individual trade size. Arrival and departure are the signs of the respective price differences. A zero price difference means unchanged, not missing.

The first backward values and final forward values remain null. Forward time is the current trade's completed hold contribution. We do not add elapsed time for the pending final row or pretend its reaction is unchanged.

In [ ]:
def step_one(frame):
    """Derive adjacent-row differences on precisely the supplied prefix."""
    return frame.with_columns(
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
step_columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
                "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(step_columns).head(15))
display(step1.select(step_columns).tail(3))

## 3. Group by exact price and the nine direction pairs

| Arrival / departure | Down (−1) | Unchanged (0) | Up (+1) |
|---|---|---|---|
| Down (−1) | Down continuation | Pause after decline | Support-like bounce |
| Unchanged (0) | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up (+1) | Resistance-like rejection | Pause after rise | Up continuation |

Each trade belongs to one cell when both directions are known. There are no visit IDs. For example, down→unchanged followed by unchanged→up is **not** merged into one bounce.

Each cell stores count, current traded volume, and sums of all six differences. Dividing a difference sum by count gives its cell mean. Signed volume changes can cancel, so they remain explanatory measurements, not an automatic strength score.

Boundary rows are displayed separately and still contribute to the total activity distribution.

In [ ]:
# These are the six differences already calculated in step one.
# Listing their names lets us apply the same sum operation to each column.
difference_columns = ["time_backward", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"]
# Keep rows with BOTH directions known so each can be assigned to one of nine groups.
# pl.col selects a column, is_not_null checks availability, and & requires both checks.
# A direction of zero (unchanged) is valid and is retained.
complete = step1.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
# Retain rows missing either direction in a separate audit table; | means OR.
# The first arrival and final departure are unknown, not unchanged.
# Keeping them separately prevents silent loss of their count and volume.
boundary_rows = step1.filter(pl.col("arrival").is_null() | pl.col("departure").is_null())
# Group by exact integer price AND arrival AND departure.
# For example, all rows at one price with arrival=-1 and departure=+1 form one group.
# Integer price units avoid floating-point equality problems. agg calculates one row per group.
profile = complete.group_by("price_units", "arrival", "departure").agg(
    # Count the trades in this group. alias gives the result its readable column name.
    # This is a trade count, not a count of independent visits.
    pl.len().alias("trade_count"),
    # Sum CURRENT trade sizes in this group, not next-trade size or volume change.
    # Thus bounce-group volume means volume on trades immediately followed by an upward move.
    pl.col("volume").sum().alias("volume_sum"),
    # Build one sum expression per difference column; * expands the list into agg arguments.
    # Forward-time sum is the group's completed hold contribution, measured in seconds.
    # Backward time is kept separately: adding the two time sums would double-count intervals.
    # Price and volume differences are signed; positive and negative values can cancel.
    # Dividing each sum by trade_count yields that group's mean difference.
    *[pl.col(name).sum().alias(name + "_sum") for name in difference_columns],
# Close the aggregation, add a dollar-price column for display, then sort the rows.
# Grouping remains on price_units; dividing by 10,000 is presentation only.
# Sorting by price and both directions makes the table easier to inspect.
).with_columns((pl.col("price_units") / 10000).alias("price")).sort("price_units", "arrival", "departure")
# Display the first 18 aggregated rows only; the full profile remains in memory.
# This limits screen output, not the data used by subsequent calculations.
display(profile.head(18))
# Report how many original trade rows were classified and how many have unknown boundaries.
# DataFrame.height is its number of rows; these two counts should total N_EVENTS.
print("Classified rows:", complete.height, "boundary rows:", boundary_rows.height)
# Show the original event identities and known directions of the boundary rows.
# select chooses display columns; it does not modify or discard the saved boundary table.
display(boundary_rows.select("event", "price", "volume", "arrival", "departure"))

# Separately aggregate ALL selected trades by price, including the boundary rows.
# Activity totals need no next-trade outcome, so their count and volume are already known.
# This table describes overall trading concentration, not a particular reaction.
activity = raw.group_by("price_units").agg(pl.len().alias("total_trades"), pl.col("volume").sum().alias("total_volume")).sort("price_units")
# Rank prices by total trade count descending; break ties by lower price first.
# Select the first price_units value to choose one reproducible example for the matrix.
# "Busiest" here means most trades, not greatest volume or strongest support.
busiest = activity.sort(["total_trades", "price_units"], descending=[True, False])["price_units"][0]
# Explicitly create all nine (arrival, departure) pairs: each direction is -1, 0 or +1.
# The departure list repeats three times, once for each arrival direction.
# Int8 matches the direction-column types in profile so the join uses compatible keys.
grid = pl.DataFrame({"arrival": [-1,-1,-1,0,0,0,1,1,1], "departure": [-1,0,1]*3}).cast(pl.Int8)
# Filter the profile to the chosen price, then match rows by both directions.
# A LEFT join preserves all nine grid rows, even if a combination never occurred.
# An unobserved combination has null aggregate values after this join.
matrix = grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
# Print the chosen price in dollars so the displayed matrix is clearly identified.
print("Nine cells at price:", busiest / 10000)
# Display directions plus count, volume and completed hold time for each of nine cells.
# Here null means NO MATCHING GROUP, so those three totals are correctly displayed as zero.
# This does not replace unknown directions or pending forward differences in the source.
# fill_null affects this display expression; the original matrix/profile remain unchanged.
display(matrix.select("arrival", "departure", pl.col("trade_count", "volume_sum", "time_forward_sum").fill_null(0)))


## 4. Find candidate prices from activity peaks
We now need a stated selection rule, not just aggregates. This first illustrative rule is:

1. Sort the **observed exact prices** from low to high.
2. A price is a local peak if its count is at least both neighboring counts and greater than at least one, **or** the same test holds for volume.
3. Require at least `MIN_TRADES` trades at that price.

Neighbors mean nearest observed prices, not a fixed tick distance. We display their gaps because the spacing can be unequal. We do not synthesize untraded prices or smooth anything. Lowest/highest observed prices are left unclassified as peaks because one neighbor is missing. Flat-topped peaks may produce several adjacent candidates; this notebook does not merge them into zones.

This is a transparent heuristic, not the only definition of a level. It can miss a rejection price with little trading. All prices and pass/fail flags are retained so the rule can be revised in a later notebook.

In [ ]:
def find_activity_peaks(table, minimum_trades):
    """Local comparisons along price, not along time."""
    result = table.sort("price_units").with_columns(
        pl.col("price_units").shift(1).alias("lower_neighbor"),
        pl.col("price_units").shift(-1).alias("upper_neighbor"),
    )
    expressions = []
    for field, label in [("total_trades", "count_peak"), ("total_volume", "volume_peak")]:
        current = pl.col(field)
        left, right = current.shift(1), current.shift(-1)
        expressions.append((left.is_not_null() & right.is_not_null() &
                            (current >= left) & (current >= right) &
                            ((current > left) | (current > right))).fill_null(False).alias(label))
    return result.with_columns(*expressions).with_columns(
        (pl.col("price_units") / 10000).alias("price"),
        ((pl.col("price_units") - pl.col("lower_neighbor")) / 10000).alias("lower_gap"),
        ((pl.col("upper_neighbor") - pl.col("price_units")) / 10000).alias("upper_gap"),
        ((pl.col("total_trades") >= minimum_trades) & (pl.col("count_peak") | pl.col("volume_peak"))).alias("candidate"),
    )

levels = find_activity_peaks(activity, MIN_TRADES)
display(levels.select("price", "total_trades", "total_volume", "lower_gap", "upper_gap", "count_peak", "volume_peak", "candidate"))
print("Activity candidates:", levels["candidate"].sum(), "of", levels.height, "observed prices")
# Show why a large-volume price may fail the minimum-trade requirement.
print("Largest volume prices, including rejected candidates:")
display(levels.sort("total_volume", descending=True).head(5).select("price", "total_trades", "total_volume", "count_peak", "volume_peak", "candidate"))


## 5. Add directional evidence without mixing it into activity
For arrivals **downward** at a price, compare down→up bounces with down→down continuations. For arrivals **upward**, compare up→down rejections with up→up continuations.

`support_share = bounce_count / (bounce_count + down_continuation_count)`

`resistance_share = rejection_count / (rejection_count + up_continuation_count)`

These are shares **conditional on an immediate nonzero departure**. Pauses are displayed separately, not silently counted as failures. Unknown or zero-denominator shares stay null. Volume shares use the corresponding current-trade volumes, not inferred buy/sell volume.

A candidate receives a support-like or resistance-like label only when that side has at least `MIN_DEPARTURES` observations and its count share reaches `MIN_REVERSAL_SHARE`. These thresholds are illustrative. Both labels can apply. Otherwise it remains an activity concentration with insufficient or mixed reaction evidence. We rank by trade count, then volume, without claiming a universal strength score.

Hold time and price/volume change magnitudes stay in the nine-cell profile. A long pause or a larger next trade is not automatically stronger support.

In [ ]:
# Attach four directly interpretable cells plus the two pause cells.
# Unchanged-arrival cells remain fully preserved in `profile`.
patterns = {"bounce": (-1,1), "down_continue": (-1,-1), "rejection": (1,-1),
            "up_continue": (1,1), "pause_down": (-1,0), "pause_up": (1,0)}
for name, (arrival, departure) in patterns.items():
    cell = profile.filter((pl.col("arrival") == arrival) & (pl.col("departure") == departure)).select(
        "price_units", pl.col("trade_count").alias(name + "_count"),
        pl.col("volume_sum").alias(name + "_volume"),
        pl.col("time_forward_sum").alias(name + "_hold"))
    levels = levels.join(cell, on="price_units", how="left").with_columns(
        pl.col(name + "_count", name + "_volume", name + "_hold").fill_null(0))

# Helper avoids inventing probabilities where there is no evidence.
def fraction(numerator, denominator):
    return pl.when(denominator > 0).then(numerator / denominator).otherwise(None)

for side, reverse, continuation in [("support", "bounce", "down_continue"), ("resistance", "rejection", "up_continue")]:
    count_total = pl.col(reverse + "_count") + pl.col(continuation + "_count")
    volume_total = pl.col(reverse + "_volume") + pl.col(continuation + "_volume")
    levels = levels.with_columns(
        count_total.alias(side + "_departures"),
        fraction(pl.col(reverse + "_count"), count_total).alias(side + "_share"),
        fraction(pl.col(reverse + "_volume"), volume_total).alias(side + "_volume_share"),
    ).with_columns(
        (pl.col("candidate") & (pl.col(side + "_departures") >= MIN_DEPARTURES) &
         (pl.col(side + "_share") >= MIN_REVERSAL_SHARE)).fill_null(False).alias(side + "_like"))

levels = levels.with_columns(
    pl.when(~pl.col("candidate")).then(pl.lit("not selected"))
    .when(pl.col("support_like") & pl.col("resistance_like")).then(pl.lit("support + resistance evidence"))
    .when(pl.col("support_like")).then(pl.lit("support-like"))
    .when(pl.col("resistance_like")).then(pl.lit("resistance-like"))
    .otherwise(pl.lit("activity concentration only")).alias("interpretation"))
candidates = levels.filter(pl.col("candidate")).sort(["total_trades", "total_volume", "price_units"], descending=[True, True, False])
summary_columns = ["price", "total_trades", "total_volume", "support_departures", "support_share", "support_volume_share",
                   "resistance_departures", "resistance_share", "resistance_volume_share", "pause_down_count", "pause_up_count", "interpretation"]
display(candidates.select(summary_columns))
if candidates.is_empty():
    print("No candidates passed the declared rule. No thresholds are relaxed automatically.")

## 6. Trace a candidate and visualize the evidence
The selected candidate below is simply the first in the explicit activity ranking. Its source rows let you check every count, volume and hold contribution yourself. If no candidate passed, we show no substitute.

Horizontal lines on the price trace are levels discovered **as of the last selected event**. They are an end-of-sample overlay, not evidence these levels were known at the beginning. Do not use this chart as a causal backtest. To inspect an earlier cutoff, rerun all calculations on that prefix.

In [ ]:
if not candidates.is_empty():
    selected = candidates["price_units"][0]
    print("Inspecting candidate price:", selected / 10000)
    display(step1.filter(pl.col("price_units") == selected).select(
        "event", "time", "price", "volume", "arrival", "departure", "time_forward", "price_backward", "price_forward", "volume_backward", "volume_forward"))
    display(profile.filter(pl.col("price_units") == selected))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)
axes[0].plot(raw["event"], raw["price"], color="black", linewidth=1)
colors = {"support-like": "green", "resistance-like": "red", "support + resistance evidence": "purple", "activity concentration only": "steelblue"}
for row in candidates.iter_rows(named=True):
    axes[0].axhline(row["price"], color=colors[row["interpretation"]], alpha=0.4, linewidth=0.8)
from matplotlib.lines import Line2D
axes[0].legend(handles=[Line2D([0], [0], color=color, label=label) for label, color in colors.items()], fontsize=8, loc="lower right")
axes[0].set(title="AAPL: first 200 trades; lines are final-prefix candidates, not historical signals", xlabel="Event", ylabel="Price")
ordered = levels.sort("price_units")
for ax, field, title in [(axes[1], "total_trades", "Trade-count distribution"), (axes[2], "total_volume", "Volume distribution")]:
    ax.plot(ordered["price"], ordered[field], ".-", linewidth=0.8, label="all observed prices")
    ax.scatter(candidates["price"], candidates[field], color="darkorange", label="selected activity peaks", zorder=3)
    ax.set(xlabel="Exact price", ylabel=field, title=title)
    if field == "total_volume":
        # A large trade can dominate linear scaling; log scaling shows small peaks too.
        ax.set_yscale("log")
        ax.set_ylabel("Total volume (log scale)")
    ax.legend()
fig.savefig(RUN / "levels.png", dpi=140)
plt.show()
plt.close(fig)

## 7. Verify accounting and save the explanation
The checks below verify the agreed differences on the hand example, all nine direction cells, local-peak selection on a small known distribution, and reconciliation back to input count/volume/time. They also verify the selected prefix cannot use event 201.

No database is changed. Every price, including rejected candidates, is retained in the output. No claim about future reaction quality is made; chronological evaluation belongs in a later notebook.

In [ ]:
# Hand example from our discussion; assert numerical differences exactly.
toy = pl.DataFrame({"time_us": [0,2000000,3000000,4000000,7000000,8000000,9000000,11000000],
                    "price_units": [100,100,101,100,101,102,101,100], "volume": [10,30,60,20,80,40,50,30]})
toy_step = step_one(toy)
assert toy_step["time_forward"].to_list() == [2.,1.,1.,3.,1.,1.,2.,None]
assert toy_step["volume_backward"].to_list() == [None,20,30,-40,60,-40,10,-20]
assert toy_step["departure"].to_list() == [0,1,-1,1,1,-1,-1,None]
# Known activity peaks are at 101 and 105, with a valley between them.
synthetic = pl.DataFrame({"price_units": [100,101,102,103,104,105,106],
                          "total_trades": [2,8,3,1,4,10,3], "total_volume": [20,90,30,10,40,120,30]})
assert find_activity_peaks(synthetic, 3).filter(pl.col("candidate"))["price_units"].to_list() == [101,105]
assert profile["trade_count"].sum() + boundary_rows.height == N_EVENTS
assert np.isclose(profile["volume_sum"].sum() + boundary_rows["volume"].sum(), raw["volume"].sum())
assert np.isclose(step1["time_forward"].sum(), (raw["time_us"][-1] - raw["time_us"][0]) / 1e6)
assert step1["departure"][-1] is None and step1["arrival"][0] is None
assert matrix.height == 9
# Full-source columns outside the prefix cannot affect its recomputed differences.
changed_source = source.with_columns(pl.when(pl.col("event") > N_EVENTS).then(pl.col("price_units") + 100000).otherwise(pl.col("price_units")).alias("price_units"))
assert step_one(changed_source.head(N_EVENTS)).equals(step1)
print("PASS: toy differences, peak rule, nine-cell matrix, count/volume/time reconciliation and prefix isolation")

raw.write_parquet(RUN / "selected_trades.parquet")
step1.write_parquet(RUN / "step_one.parquet")
profile.write_parquet(RUN / "nine_direction_profile.parquet")
levels.write_parquet(RUN / "all_prices_and_decisions.parquet")
candidates.write_parquet(RUN / "candidate_levels.parquet")
NOTEBOOK = REPO / "research/level_book/v8/02_price_concentration_reaction_levels.ipynb"
manifest = dict(source_run=str(SOURCE_RUN), source_sha256=source_hash, ticker="AAPL", day="2026-08-20",
                selected_events=N_EVENTS, original_events=source.height, unique_prices=activity.height,
                candidate_count=candidates.height, classifications=candidates.group_by("interpretation").len().to_dicts(),
                thresholds=dict(min_trades=MIN_TRADES, min_departures=MIN_DEPARTURES, min_reversal_share=MIN_REVERSAL_SHARE),
                as_of_us=raw["time_us"][-1], last_ordinal=raw["ordinal"][-1],
                notebook_sha256=hashlib.sha256(NOTEBOOK.read_bytes()).hexdigest(),
                git_commit=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip(),
                python=sys.version, numpy=np.__version__, polars=pl.__version__, checks="passed")
(RUN / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(RUN / "source_provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))